# 05 · Oracle labels and the evaluator comparison (Table 6)

**Runtime: CPU is fine.** Everything reads the caches on Drive (generations from notebook 01, embeddings from its free-text cell).

| Part | What it does | Output on Drive (`MyDrive/mpac_hi_pt/`) |
|---|---|---|
| A · Oracle sheets | Samples generations and writes one sheet per annotator, per language | `oracle/<round>/<lang>/<annotator>.csv` |
| A2 · LLM panel | No human annotators: Claude Opus and Llama-3.3-70B fill the sheets, Claude Sonnet adjudicates | `oracle/<round>/<lang>/filled/<model>.csv` + `.jsonl` |
| B · Agreement | Reads the filled sheets, Cohen's kappa per language | (printed) |
| C · Table 6 | Thresholds tuned on **tune** labels, then systems 4, 5, 7 on the same outputs and budget, scored on **test** labels | `results/<model>/<checkpoint>/table6_<source>.json` |

Protocol: `docs/oracle_protocol.md`. Order: **pilot** round (30 rows, agreement only, never used as labels), then **main** round. Part C can run before any human labels with `SOURCE = "proxy"` (TruthfulQA answers, honesty only) to check the pipeline; those numbers are **not reportable**.

System 6 (LLM-as-judge) isn't in the table yet (D5). Its judge must not be a member of the LLM panel.

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[signals]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
QUICK = False
MODEL = "qwen2.5-7b"
CONFIGS = ["configs/hi_pt.yaml"] + (["configs/hi_pt_quick.yaml"] if QUICK else [])

from google.colab import drive
drive.mount("/content/drive")
from mpac import load_config
from mpac.config import get_model
from mpac.paths import resolve_root, subdir
cfg = load_config(*CONFIGS)
m = get_model(cfg, None if QUICK else MODEL)
print("model:", m["id"], "| languages:", cfg["languages"], "| artifacts ->", resolve_root(cfg))

## A · Oracle sheets

Run once per round: first `ROUND = "pilot"` (30 rows), then, once the pilot's kappa is ≥ 0.6, `ROUND = "main"` (`data.n_oracle_labels`, 120 for hi/pt). The main round never re-draws pilot rows, and an existing round is never overwritten.

- `ANNOTATORS`: two per language, fluent in English and that language. Use short names without spaces; each gets their own file (and their own row order).
- `LABEL_CHECKPOINTS`: **base only** for the hi/pt pilot. 120 labels spread over three checkpoints would leave about 4 test labels per cell per checkpoint, too few to score anything.

**Send each annotator** their CSV (Drive → right-click → Share, or download it) and `docs/oracle_protocol.md`. They fill `aligned` and `collapsed_vs_reference` with yes / no (and `notes` if needed), then **download as CSV** (File → Download → .csv in Google Sheets) and you put it back as `oracle/<round>/<lang>/filled/<annotator>.csv`, same file name. The adjudicator's sheet goes in the same folder as `adjudication.csv`.

**No human annotators?** Use the LLM panel (A2): `ROUND = "llm_pilot"` then `"llm_main"`. The annotator names come from `oracle.llm_panel` and the main LLM round labels **every** remaining row, not just 120.

In [ ]:
from mpac.config import all_languages
from mpac.data.loaders import load_items
from mpac.data.oracle import export_round, round_keys, sample_for_labelling
from mpac.data.splits import get_or_create_splits
from mpac.generation.cache import GenerationCache

ROUND = "pilot"   # "pilot" (30 rows) first, then "main"; LLM panel: "llm_pilot", then "llm_main"
ANNOTATORS = {"hi": ["hi_annotator1", "hi_annotator2"], "pt": ["pt_annotator1", "pt_annotator2"]}
if ROUND.startswith("llm"):
    ANNOTATORS = {lang: list(cfg["oracle"]["llm_panel"]["annotators"]) for lang in cfg["languages"]["targets"]}
LABEL_CHECKPOINTS = ["base"]

items = load_items(cfg)
splits = get_or_create_splits(cfg, items)
cache = GenerationCache(subdir(cfg, "generations"))
gens = {(m["id"], ck): cache.load_all(m["id"], ck, all_languages(cfg)) for ck in LABEL_CHECKPOINTS}
oracle_dir = subdir(cfg, "oracle")
earlier = set().union(*(round_keys(d) for d in oracle_dir.iterdir() if d.is_dir() and d.name != ROUND)) \
    if oracle_dir.exists() else set()
n = 30 if ROUND.endswith("pilot") else 10**9 if ROUND.startswith("llm") else cfg["data"]["n_oracle_labels"]
rows = sample_for_labelling(items, gens, n, cfg["languages"]["targets"], cfg["languages"]["reference"],
                            splits, cfg["project"]["seed"], exclude=earlier)
paths = export_round(rows, oracle_dir / ROUND, ANNOTATORS, cfg["project"]["seed"])
from collections import Counter
print(len(rows), "rows, excluding", len(earlier), "from earlier rounds:",
      dict(Counter((r["lang"], r["objective"]) for r in rows)))
for lang, ps in paths.items():
    print(lang, [str(p) for p in ps])

## B · Agreement

After both annotators of a language have put their sheets in `filled/`. Continue to the main round only if kappa ≥ `oracle.min_kappa` (0.6) on **both** fields; otherwise discuss the disagreements, tighten `docs/oracle_protocol.md` (log the change at the bottom) and run a new pilot round (`ROUND = "pilot2"`).

In [ ]:
from mpac.data.oracle import agreement, load_round, resolve

ROUND = "pilot"
for lang, (labels, adj) in load_round(subdir(cfg, "oracle") / ROUND).items():
    if not labels:
        print(lang, ": no filled sheets yet"); continue
    a = agreement(labels)
    _, unresolved = resolve(labels, adj)
    ok = all(v["kappa"] >= cfg["oracle"]["min_kappa"] for v in a.values())
    print(f"{lang}: {len(labels)} labels from {len({l.annotator for l in labels})} annotators | "
          f"unresolved {len(unresolved)} | {'OK' if ok else 'BELOW min_kappa'}")
    for field, v in a.items():
        print(f"   {field:24s} kappa {v['kappa']:.2f}  raw agreement {v['raw_agreement']:.2f}  n {v['n_items']:.0f}")

## A2 · LLM panel (no human annotators)

Run part A first with `ROUND = "llm_pilot"` (30 rows), then again with `"llm_main"` (every other row). Then for each round:

1. **Claude Opus** (any runtime, CPU is fine). Needs `ANTHROPIC_API_KEY` in Colab secrets (key icon on the left → add it, allow notebook access). About 1,000 rows for the base checkpoint, roughly $30–40 at effort `high`.
2. **Llama-3.3-70B** (**A100** runtime: 4-bit is about 40 GB). Needs `HF_TOKEN` in Colab secrets and the Llama 3.3 licence accepted on its Hugging Face page. Re-run the setup cells after switching runtime.
3. **Claude Sonnet** adjudicates only the rows Opus and Llama disagree on.
4. Part B for agreement (set its `ROUND`), then the known-answer check below.

Every reply is logged to `filled/<model>.jsonl`. If a cell stops or some rows error, run it again: only the missing rows are sent.
If the pilot's kappa is below 0.6, read the disagreements (`notes` column = each model's evidence) before running the main round. If the cause is the rubric, change `docs/oracle_protocol.md` **and** `ORACLE_SYSTEM` in `data/llm_oracle.py` together and run `llm_pilot2`.

In [ ]:
!pip install -q -e ".[llm,gpu]"   # gpu: Llama in 4-bit
import os
from google.colab import userdata
for k in ("ANTHROPIC_API_KEY", "HF_TOKEN"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print(k, "not in Colab secrets (needed for", "Claude)" if k.startswith("ANTH") else "Llama)")
from mpac.data.llm_oracle import adjudicate, anthropic_annotator, hf_annotator, known_answer_check, label_round

ROUND = "llm_pilot"
panel = cfg["oracle"]["llm_panel"]
round_dir = subdir(cfg, "oracle") / ROUND

In [ ]:
# 1 · Claude Opus
name, model_id = "opus", panel["annotators"]["opus"]
print(label_round(round_dir, name, anthropic_annotator(model_id, effort=panel["effort"])))

In [ ]:
# 2 · Llama-3.3-70B (A100 runtime)
name, model_id = "llama", panel["annotators"]["llama"]
print(label_round(round_dir, name, hf_annotator(model_id)))

In [ ]:
# 3 · Claude Sonnet adjudicates the disagreements
(name, model_id), = panel["adjudicator"].items()
print(adjudicate(round_dir, name, anthropic_annotator(model_id, effort=panel["effort"])))

In [ ]:
# 4 · Known-answer check: TruthfulQA rows, where the right option is known without a human.
# aligned_accuracy: aligned == the target's top option is correct. lost_recall: rows right in English and
# wrong in the target flagged as collapse. same_specificity: rows with the same option in both not flagged.
import pandas as pd
from mpac.data.loaders import load_items
print(pd.DataFrame(known_answer_check(round_dir, load_items(cfg))).T.round(3).to_string())

## C · Table 6: evaluator systems on the same outputs

- `SOURCE = "proxy"` works today (pipeline check, **not reportable**); `"human"` reads the resolved `main` round, `"llm"` the resolved `llm_main` round (report it as an LLM-panel oracle, with its kappa and known-answer check).
- Thresholds start at each language's LaBSE floor + 0.15 (options: 0.3) and are tuned per (language, signal) on **tune** labels only; the table shows which were tuned.
- The adaptive run sets the budget; the single evaluator and the fixed team get the same number of evaluations. Scores use **test** labels and count labelled items a system never looked at as missed.
- `wrong_language_responses`: free-text responses fastText says aren't in the prompt's language. Agents flag those as collapse.

In [ ]:
import pandas as pd
from mpac.analysis.evaluators import table6

SOURCE = "proxy"        # "human" once the main round is resolved, "llm" for the LLM panel
ROUND = {"human": "main", "llm": "llm_main"}.get(SOURCE, "main")
CHECKPOINT = "base"

out = table6(cfg, m["id"], CHECKPOINT, source=SOURCE, round_name=ROUND)
print(out["oracle"], "\nbudget:", out["budget"], "evaluations | wrong-language responses:",
      out["wrong_language_responses"], "\nsaved ->", out["path"], "\n")
cols = ["precision", "recall", "f1", "n_labelled", "n_labelled_evaluated", "n_evaluations", "cells", "tokens"]
print(pd.DataFrame(out["systems"]).T[cols].round(3).to_string(), "\n")
print(pd.DataFrame(out["thresholds"]).T[["threshold", "source", "n_labels", "n_collapsed"]].to_string(), "\n")
a = out["adaptive"]
print("adaptive population by cycle:", a["population_by_cycle"], "| ops:", a["lifecycle_ops"])
print("final agents:", a["final_alive"])

### Review the verdicts by hand

`table6()` also writes `results/<model>/<checkpoint>/table6_<source>_review.csv`: one row per (language, item) with both responses, the signal, the divergence and threshold, the oracle label (blank if unlabelled), and per system whether it flagged the item and how many times it looked. Open it in Google Sheets (Drive → right-click → Open with) and filter on `systems_differ`, `oracle_collapsed` or a system's `flagged` column. The cell below prints the summary.

If `systems_differ` is never true, the systems handed in the same answers. The oracle only grades them, so identical scores mean identical verdicts, not a shared oracle.

In [ ]:
rev = pd.read_csv(out["review_path"])
flag_cols = [c for c in rev.columns if c.endswith("| flagged")]
print("pairs:", len(rev), "| pairs where systems differ:", out["n_pairs_systems_differ"])
print("\nflagged per system (all pairs, labelled or not):")
print(rev[flag_cols].eq(True).sum().to_string())
print("\nevaluations per pair:")
print(rev[[c.replace("flagged", "n_evals") for c in flag_cols]].describe().loc[["min", "mean", "max"]].round(1).to_string())
lab = rev[rev["oracle_collapsed"].notna()]
print("\nlabelled pairs: flagged vs oracle (first system), by split")
print(pd.crosstab([lab["split"], lab["oracle_collapsed"]], lab[flag_cols[0]].eq(True), rownames=["split", "oracle"], colnames=["flagged"]))
cols = ["lang", "item_id", "objective", "oracle_collapsed", "signal", "divergence", "threshold", *flag_cols]
print("\nmissed collapses (oracle yes, not flagged):")
print(lab[lab["oracle_collapsed"].eq(True) & ~lab[flag_cols[0]].eq(True)][cols].head(10).to_string(index=False))
print("\nfalse alarms (oracle no, flagged):")
print(lab[lab["oracle_collapsed"].eq(False) & lab[flag_cols[0]].eq(True)][cols].head(10).to_string(index=False))
if out["n_pairs_systems_differ"]:
    print("\nwhere systems differ:")
    print(rev[rev["systems_differ"]][cols].head(20).to_string(index=False))